<a href="https://colab.research.google.com/github/Jeevan0221/Tj-ai-portfolio/blob/main/day9_ai_data_analyst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
!pip install anthropic
import anthropic
from google.colab import userdata
print("all tools loaded successfully!")

all tools loaded successfully!


In [50]:
api_key = userdata.get('ANTHROPIC_KEY')
client = anthropic.Anthropic(api_key=api_key)

print("connected to claude")

connected to claude


In [54]:
data = {
    'years_experience': [1,2,3,4,5,6,7,8,9,10],
    'ai_skill'        : [0,0,1,1,1,1,1,1,1,1],
    'salary'          : [60000,70000,80000,90000,100000,
                         110000,120000,130000,140000,150000]
}

df = pd.DataFrame(data)

#Split and train
X = df[['years_experience', 'ai_skill']]
y = df['salary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Train the model
model = LinearRegression()
model.fit(X_train, y_train)

#Get predictons and error
predictions = model.predict(X_test)
error = mean_absolute_error(y_test, predictions)

print(f"Model trained! Average prediction error: ${error:,.0f}")



Model trained! Average prediction error: $0


In [55]:
#Extract what the model learned
experience_impact = model.coef_[0]
ai_skill_impact = model.coef_[1]
base_salary = model.intercept_

#predict TJ's Salary

my_profile = pd.DataFrame({'years_experience':[8], 'ai_skill': [1]})
my_salary = model.predict(my_profile)[0]

print(f"Every year of experience adds: ${experience_impact:,.0f}")
print(f"Having AI skill adds: ${ai_skill_impact:,.0f}")
print(f"Base salary(0 experience): ${base_salary:,.0f}")
print(f"TJ's Predicted salary: ${my_salary:,.0f}")

Every year of experience adds: $10,000
Having AI skill adds: $-0
Base salary(0 experience): $50,000
TJ's Predicted salary: $130,000


In [53]:
analysis_prompt = f"""
You are a data analyst. Analyze these ML model results about tech salaries:

DATASET: 10 tech employees, features: years of experience and AI skill (yes/no)

MODEL FINDINGS:
- Base salary (0 experience, no AI skill): ${base_salary:,.0f}
- Each year of experience adds: ${experience_impact:,.0f}
- AI skill coefficient: ${ai_skill_impact:,.0f}
- Model average error: ${error:,.0f}

CANDIDATE PROFILE:
- Name: TJ
- Years of experience: 8
- Has AI skill: Yes (currently learning)
- Predicted salary: ${my_salary:,.0f}

Please explain:
1. What the model learned about salary patterns
2. Why the AI skill coefficient might look unusual
3. What this means for TJ's career and salary potential
4. One recommendation for TJ

Keep it simple and clear, no technical jargon.
"""

message = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": analysis_prompt}
    ]
)

print("=== AI Data Analyst Report ===")
print(message.content[0].text)

=== AI Data Analyst Report ===
# Analysis of TJ's Salary Prediction

## 1. What the Model Learned

The model identified two main salary drivers:
- **Experience is valuable**: Each year adds roughly $14K to salary
- **AI skills showed a negative pattern** in this particular dataset

For someone with 0 experience and no AI skills, the baseline is $46K.

## 2. Why the AI Skill Coefficient Looks Unusual

The **-$11,557 seems backwards** because we'd expect AI skills to boost pay. Here's what likely happened:

- **Small dataset problem**: With only 10 employees, the pattern can be misleading
- **Possible explanation**: Maybe junior AI learners in this company happened to earn less (perhaps they're newer hires who are learning on the job), while experienced non-AI specialists earn more
- **This doesn't reflect reality**: In the broader tech market, AI skills typically command higher salaries

This is a **red flag** that this model shouldn't be trusted for major career decisions.

## 3. What 